# Pre-class: Monday Morning — The Bar to Beat
**⏱ This pre-class notebook takes approximately 15 minutes.**

---

## Scenario: Monday — Marcus's New Brief

It's the start of week 4 at NorthStar Retail. Friday last week Sarah showed Marcus the L03 logistic regression. He nodded, then said:

> *"This is the first model we own. But we're hitting capacity at 200 calls. Can you squeeze MORE recall without more false positives? Try those tree-based models you mentioned."*

This morning Sarah sits down with two coffees, the same NorthStar churn dataset from L03, and a clear goal: **beat the L03 baseline.**

By Friday she has to show:
1. A trained Random Forest classifier.
2. A trained Gradient Boosting classifier.
3. Tuned versions of both — with cross-validated evidence.
4. A recommendation on which model NorthStar should put into production.

Today is just the setup: **re-train the L03 baseline so we see the bar, then try a single decision tree to see why "one tree isn't enough."**

**By the end of this notebook you will be able to:**
- State the L03 baseline F1 we're trying to beat
- See a single deep decision tree overfit dramatically
- Understand why we need an *ensemble* of trees, not just one

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — ready to compare baseline vs trees")

## Step 1 — Reload the dataset and the L03 pipeline

We use the SAME data and the SAME train/test split as L03 so the comparison is honest.

In [ ]:
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])
print(f"Training set: {len(X_train):,} customers ({y_train.mean():.1%} churn)")
print(f"Test set:     {len(X_test):,} customers ({y_test.mean():.1%} churn)")

## Step 2 — Re-train the L03 logistic regression baseline

This is the model Sarah presented to Marcus last Friday. We re-fit it on the same training set so we have an apples-to-apples F1 to compare against.

In [ ]:
logreg_pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
logreg_pipe.fit(X_train, y_train)

# Predict probabilities, apply the threshold Sarah recommended in L03
y_proba_lr = logreg_pipe.predict_proba(X_test)[:, 1]
y_pred_lr  = (y_proba_lr >= 0.25).astype(int)

print("=== L03 baseline (LogisticRegression, threshold = 0.25) ===")
print(f"Accuracy:   {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision:  {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall:     {recall_score(y_test, y_pred_lr):.3f}")
print(f"F1:         {f1_score(y_test, y_pred_lr):.3f}")
print(f"Flagged:    {int(y_pred_lr.sum())} customers")
print()
print(f"→ This is THE BAR. Every model this week has to beat F1 ≈ {f1_score(y_test, y_pred_lr):.3f}.")

## Step 3 — One naive decision tree

Now let's try the simplest tree-based model: a single `DecisionTreeClassifier` grown to full depth. Note we still wrap it in the same `Pipeline` — the only change is the final step.

(Trees don't actually need scaling — but we keep it in the Pipeline so the comparison is clean. Trees ignore the scaling step's effect.)

In [ ]:
tree_pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", DecisionTreeClassifier(random_state=42)),  # default: grow until pure
])
tree_pipe.fit(X_train, y_train)

# Tree's predict() uses threshold=0.5 internally on probabilities derived from leaf purity
y_pred_tree_train = tree_pipe.predict(X_train)
y_pred_tree_test  = tree_pipe.predict(X_test)

print("=== Single decision tree (full depth) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_tree_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_tree_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_tree_train) - accuracy_score(y_test, y_pred_tree_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST recall:{recall_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST prec:  {precision_score(y_test, y_pred_tree_test):.3f}")

### 💡 What you should notice

- **Training accuracy is essentially 1.00.** The tree perfectly classifies the training set — it has memorised every customer.
- **Test accuracy is much lower.** The gap is the variance / overfitting symptom.
- **The F1 may actually be HIGHER than logistic regression** at threshold 0.5 — the tree's leaves are individually predictive. But this is brittle: re-run on a different split and the F1 will swing wildly.

This is the textbook overfitting failure mode. One tree memorises; it doesn't generalise.

**The fix isn't to make the tree shallower.** A shallow tree underfits — it can't capture interactions. The fix is to train MANY trees on slightly different subsets and AVERAGE them. That's bagging. That's Random Forest. That's Tuesday's notebook.

## Step 4 — A first shallow tree (just to show the bias-variance tradeoff)

For comparison, here's what happens if we constrain the tree to `max_depth=3`. It can't overfit — but it can't fit much, either.

In [ ]:
shallow_tree = Pipeline([
    ("prep",  preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])
shallow_tree.fit(X_train, y_train)

y_pred_st_train = shallow_tree.predict(X_train)
y_pred_st_test  = shallow_tree.predict(X_test)

print("=== Shallow tree (max_depth=3) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_st_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_st_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_st_train) - accuracy_score(y_test, y_pred_st_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_st_test):.3f}")
print()
print("→ Train and test agree (no overfitting).")
print("→ But the F1 is lower than the deep tree's TEST F1 — the shallow tree can't capture the patterns.")
print()
print("This is the BIAS-VARIANCE tradeoff in one notebook:")
print("  Deep tree:    low bias, HIGH variance (overfits)")
print("  Shallow tree: HIGH bias, low variance (underfits)")
print("  Random Forest will give us low bias AND low variance — by averaging many deep trees.")

## ✅ Section Summary

| Model | Train acc | Test acc | Test F1 | Diagnosis |
|---|---|---|---|---|
| **L03 Logistic Regression @ 0.25** | — | ~0.84 | ~0.28 | The bar to beat |
| **Single decision tree (full depth)** | ~1.00 | ~0.79 | varies | OVERFITS — memorises training data |
| **Shallow tree (max_depth=3)** | ~0.88 | ~0.88 | very low | UNDERFITS — too simple |

**Key insight:**
> One tree alone is never the answer. Either it overfits (full depth) or it underfits (shallow). Random Forest fixes this by AVERAGING many deep trees, each trained on slightly different data. The variance washes out; the bias stays low.

**Bring to class:**
1. The L03 baseline F1 (the bar to beat).
2. The train-vs-test gap of the deep tree (the overfitting symptom).
3. One question about how Random Forest changes this picture.

---
**In class → Open `02_decision_tree_to_forest.ipynb` first.** That notebook builds the Random Forest properly.